# 01 - Day 1: INDB tables + Khana-to-INDB mapping
Run the cells top to bottom. **Do not save this notebook back to GitHub** (its outputs contain INDB rows).
All results are saved to Google Drive: `MyDrive/AaharAI/data/`.

In [ ]:
GITHUB_USER = "YOUR-GITHUB-USERNAME"   # <-- EDIT THIS (the account that owns the AaharAI repo)

from google.colab import drive
drive.mount("/content/drive")
!pip -q install rapidfuzz
import sys
!git -C /content/AaharAI pull || git clone https://github.com/{GITHUB_USER}/AaharAI.git /content/AaharAI
sys.path.insert(0, "/content/AaharAI")
%load_ext autoreload
%autoreload 2

## Step 1: put INDB on Drive (once) and build the tables

In [ ]:
from pathlib import Path
DATA = Path("/content/drive/MyDrive/AaharAI/data"); DATA.mkdir(parents=True, exist_ok=True)
!test -f {DATA}/indb/INDB.xlsx || git clone https://github.com/lindsayjaacks/Indian-Nutrient-Databank-INDB.git {DATA}/indb
print("INDB files:", sorted(p.name for p in (DATA / "indb").glob("*.xlsx")))

In [ ]:
from src import indb
rec, kept, nutrition = indb.build_tables()
print("ingredient rows:", len(rec), "| kept recipes:", len(kept), "| nutrition rows:", len(nutrition))
print(rec.cat.value_counts().to_string())
print("duplicate recipe codes in kept:", kept.recipe_code.duplicated().sum())
print("guessed serving units:", int(nutrition.serving_guessed.sum()), "| no serving info:", int(nutrition.no_of_servings.isna().sum()))
print("\nMost common unclassified ('other') ingredients:")
print(rec[rec.cat == "other"].canonical.value_counts().head(25).to_string())

In [ ]:
# sanity check: per-serving values should look like one serving of the stated unit
print(nutrition.sample(5, random_state=0)[["recipe_name", "serving_unit", "size_of_servings", "kcal_100g", "kcal", "serving_note"]].to_string(index=False))

## Step 2: Khana-to-INDB mapping, first batch (clear matches)
This never overwrites an existing `mapping.csv`, so your manual edits are safe.

In [ ]:
import pandas as pd
from src.khana_classes import KHANA_CLASSES

M = {  # label: (indb_recipe_id, status, note)
 "banana chips": ("OSR119", "match", ""),
 "chapati": ("ASC096", "match", "Chapati/Roti"),
 "dhokla": ("ASC474", "match", "alts BFP440, BFP573, OSR115"),
 "fish curry": ("ASC246", "match", "alt BFP223 Bengal fish curry"),
 "gajar ka halwa": ("ASC295", "match", ""),
 "garlic naan": ("ASC142", "variant", "plain naan, garlic not in recipe"),
 "gujhia": ("ASC280", "match", "INDB spells it Gunjia"),
 "gulab jamun": ("ASC348", "match", "with khoya; alt BFP386"),
 "idli": ("ASC144", "match", "alts BFP147, BFP579"),
 "khakhra": ("OSR102", "match", ""),
 "kheer": ("ASC282", "match", "rice kheer"),
 "kulfi": ("ASC321", "match", ""),
 "masala dosa": ("ASC146", "match", ""),
 "medu vada": ("BFP436", "match", "plain urad dal vada; medu vada is in its alternative names"),
 "moong dal halwa": ("ASC299", "match", ""),
 "murukku": ("OSR113", "match", "rice murukku"),
 "pav bhaji": ("OSR112", "match", ""),
 "phirni": ("ASC292", "match", ""),
 "poha": ("BFP044", "match", "alt BFP045 vegetable poha"),
 "rasgulla": ("BFP392", "match", ""),
 "rava dosa": ("ASC147", "match", "semolina dosa"),
 "sabudana vada": ("ASC379", "match", "peanut sago vada"),
 "samosa": ("ASC361", "variant", "potato samosa"),
 "uttapam": ("BFP152", "match", "alt ASC148"),
}
path = DATA / "mapping.csv"
if path.exists():
    print("mapping.csv already exists, not overwriting:", path)
    print(pd.read_csv(path).status.value_counts().to_string())
else:
    rows = [(l, *M.get(l, ("", "pending", ""))) for l in KHANA_CLASSES]
    pd.DataFrame(rows, columns=["khana_label", "indb_recipe_id", "status", "notes"]).to_csv(path, index=False)
    print("created", path, "|", sum(r[2] != "pending" for r in rows), "filled, 56 pending")

## Step 3: keyword search for the remaining classes
Send the printed output back. A class is 'none' only if nothing sensible shows up here.

In [ ]:
nut = pd.read_csv(DATA / "nutrition.csv")
KEYS = {
 "aloo gobi": ["gobi", "cauliflower"], "aloo methi": ["methi"], "aloo mutter": ["matar", "mutter", "peas"],
 "aloo paratha": ["parantha", "paratha"], "amritsari kulcha": ["kulcha"], "anda curry": ["egg", "anda", "ande"],
 "bhindi masala": ["bhindi", "okra", "lady"], "biryani": ["biryani", "biriyani"],
 "boondi laddu": ["ladoo", "laddu", "boondi"], "chaas": ["buttermilk", "chaas", "lassi"],
 "chole bhature": ["chole", "chhole"], "dal khichdi": ["khichdi", "khichri"],
 "hara bhara kabab": ["kabab", "kebab"], "idiyappam": ["idiyappam", "idiappam", "noodle", "sevai"],
 "kaju katli": ["kaju", "cashew", "burfi", "barfi"], "misal pav": ["misal", "usal", "sprout"],
 "navratan korma": ["korma", "navratan"], "onion pakoda": ["pakoda", "pakora", "bhajia", "fritter"],
 "palak paneer": ["palak", "spinach"], "pani puri": ["pani", "golgappa", "puchka"],
 "papdi chaat": ["chaat", "chat"], "pongal": ["pongal"], "puri bhaji": ["poori", "bhaji"],
 "rajma chawal": ["rajma", "rajmah"], "sabudana khichdi": ["sabudana", "sago"],
 "sev puri": ["sev"], "steamed momo": ["momo", "dumpling"], "vada pav": ["pav"],
 "chikki": ["chikki", "peanut"], "besan laddu": ["ladoo", "laddu"], "grilled sandwich": ["sandwich"],
 "seekh kebab": ["seekh"], "paneer masala": ["paneer"], "margherita pizza": ["pizza"],
 "chivda": ["chivda", "chiwda"], "jalebi": ["jalebi", "jilebi"], "masala papad": ["papad"],
 "modak": ["modak"], "mysore pak": ["mysore"], "falooda": ["falooda"], "dabeli": ["dabeli"],
 "ghevar": ["ghevar"], "paniyaram": ["paniyaram", "appe"], "balushahi": ["balushahi", "badusha"],
 "patrode": ["patra", "colocasia", "arbi"], "solkadhi": ["kokum", "kadhi"], "thukpa": ["thukpa"],
 "garlic bread": ["garlic bread"], "chicken wings": ["wing"],
}
for label, kws in KEYS.items():
    m = nut[nut.recipe_name.str.lower().str.contains("|".join(kws), na=False)]
    print(f"\n{label.upper()} ({len(m)} hits)")
    if len(m):
        print(m[["recipe_code", "recipe_name", "no_of_servings"]].head(8).to_string(index=False))

## Step 4: ingredient lists for the pizza decision

In [ ]:
for code in ["ASC376", "BFP122"]:
    print(code, nut[nut.recipe_code == code].recipe_name.tolist())
    print(rec[rec.recipe_code == code][["ingredient_name_org", "amount", "unit"]].to_string(index=False), "\n")